# Data Cleaning - Capa SILVER

En este notebook se realiza la transformación y limpieza de los datos para la capa SILVER.

El objetivo es preparar la información proveniente de la capa BRONZE para su posterior análisis y explotación.

In [ ]:
import os
import sys

# Configurar variables de entorno para Hadoop en Windows
HADOOP_HOME = os.environ.get("HADOOP_HOME", "C:\\hadoop")

os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["hadoop.home.dir"] = HADOOP_HOME
os.environ["PATH"] = f"{os.environ.get('PATH', '')};{HADOOP_HOME}\\bin"

# Configurar Python para Spark
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Configurar memoria antes de iniciar Spark
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"

In [ ]:
# Importar librerías

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lower,
    regexp_replace,
    to_timestamp,
    coalesce,
    try_to_timestamp,
    lit,
)

## Crear la sesión de Apache Spark

Se crea una sesión de Spark para el procesamiento distribuido de datos.

In [ ]:
# Crear sesión de Spark
spark = (
    SparkSession.builder.appName("FinancialDigitalTwin_Silver")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

# Comprobar la memoria disponible para Java
memoria_java_gb = spark.sparkContext._jvm.java.lang.Runtime.getRuntime().maxMemory() / (
    1024**3
)

print("Spark inicializado")
print(f"Memoria máxima de Java: {memoria_java_gb:.1f} GB")

## Cargar los conjuntos de datos

In [ ]:
# Cargar datasets

INPUT_PATH = "../data/processed/bronze"
OUTPUT_PATH = "../data/processed/silver"

# Crear directorio SILVER si no existe

import os

os.makedirs(OUTPUT_PATH, exist_ok=True)

users_spark = spark.read.parquet(f"{INPUT_PATH}/users.parquet")

cards_spark = spark.read.parquet(f"{INPUT_PATH}/cards.parquet")

transactions_spark = spark.read.parquet(f"{INPUT_PATH}/transactions.parquet")

print("Datos cargados")

In [ ]:
# Importar funciones adicionales

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Cargar cuentas y obligaciones

accounts_spark = spark.read.parquet(f"{INPUT_PATH}/accounts.parquet")

obligations_spark = spark.read.parquet(f"{INPUT_PATH}/obligations_payments.parquet")

## Explorar la información

In [ ]:
# Mostrar primeras filas

users_spark.show(5)
cards_spark.show(5)
transactions_spark.show(5)

In [ ]:
# Mostrar esquema

users_spark.printSchema()
cards_spark.printSchema()
transactions_spark.printSchema()

## Transformar los datos

In [ ]:
# Transformar users

from functools import reduce


# Conservar la fila más completa por identificador
def conservar_completo(df, clave, columnas=None):
    campos = columnas or [c for c in df.columns if c != clave]

    completos = reduce(
        lambda a, b: a + b, [F.col(c).isNotNull().cast("int") for c in campos], F.lit(0)
    )

    contenido = F.to_json(
        F.struct(*[F.col(c) for c in sorted(df.columns)]),
        options={"ignoreNullFields": "false"},
    )

    ventana = Window.partitionBy(clave).orderBy(
        completos.desc(), F.sha2(contenido, 256).asc()
    )

    return (
        df.withColumn("_position", F.row_number().over(ventana))
        .filter(F.col("_position") == 1)
        .drop("_position")
    )


users_silver = users_spark

for column in ["gender", "email"]:
    users_silver = users_silver.withColumn(column, F.lower(F.trim(F.col(column))))

for column in ["full_name", "address", "city", "state", "country", "postal_code"]:
    users_silver = users_silver.withColumn(column, F.trim(F.col(column)))

for column in ["yearly_income", "per_capita_income", "total_debt"]:
    users_silver = users_silver.withColumn(column, F.col(column).cast("double"))

users_silver = conservar_completo(users_silver, "id")


# Validar identidad y correo antes de relacionar los datasets
identidad_invalida = (
    F.col("id").isNull()
    | (F.col("id") < 0)
    | F.col("full_name").isNull()
    | (F.length("full_name") == 0)
    | F.col("email").isNull()
    | ~F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
)

if users_silver.filter(identidad_invalida).limit(1).count():
    raise ValueError("Hay usuarios con identificador, nombre o correo inválido.")

if users_silver.groupBy("email").count().filter("count > 1").limit(1).count():
    raise ValueError("Un correo está relacionado con más de un cliente.")

## Separar registros rechazados

Se identifican los clientes existentes a partir de los usuarios de SILVER.

Se separan las tarjetas con identificador nulo o negativo y las tarjetas
o transacciones que no corresponden a un cliente existente.

Los registros rechazados se guardan por separado, indicando el motivo.

In [ ]:
# Eliminar duplicados de usuarios antes de relacionar los registros

users_silver = users_silver.dropDuplicates(["id"])


# Identificar clientes existentes

clientes = (
    users_silver.filter(col("id").isNotNull() & (col("id") >= 0))
    .select(col("id").alias("client_id"))
    .distinct()
)


# Separar tarjetas con ID inválido

cards_invalid_id = cards_spark.filter(col("id").isNull() | (col("id") < 0)).withColumn(
    "rejection_reason", lit("ID de tarjeta nulo o negativo")
)

cards_valid_id = cards_spark.filter(col("id").isNotNull() & (col("id") >= 0))


# Separar tarjetas sin cliente existente

cards_without_client = cards_valid_id.join(
    clientes, "client_id", "left_anti"
).withColumn("rejection_reason", lit("Cliente inexistente"))

cards_rejected = cards_invalid_id.unionByName(cards_without_client)

cards_valid = cards_valid_id.join(clientes, "client_id", "left_semi")


# Separar transacciones sin cliente existente

transactions_rejected = transactions_spark.join(
    clientes, "client_id", "left_anti"
).withColumn("rejection_reason", lit("Cliente inexistente"))

transactions_valid = transactions_spark.join(clientes, "client_id", "left_semi")


# Mostrar cantidad de rechazados

print("Tarjetas rechazadas:", cards_rejected.count())

print("Transacciones rechazadas:", transactions_rejected.count())


# Guardar rechazados de la ejecución actual

cards_rejected.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/rejected/cards.parquet")

transactions_rejected.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/rejected/transactions.parquet"
)

## Transformar los registros aceptados

Se normalizan los textos y los tipos numéricos de las tarjetas y
transacciones aceptadas.

Los registros rechazados permanecen separados del procesamiento.

In [ ]:
# Transformar cards

cards_silver = cards_valid.withColumn("card_brand", trim(lower(col("card_brand"))))

cards_silver = cards_silver.withColumn("card_type", trim(lower(col("card_type"))))

cards_silver = cards_silver.withColumn("has_chip", trim(lower(col("has_chip"))))

cards_silver = cards_silver.withColumn(
    "card_on_dark_web", trim(lower(col("card_on_dark_web")))
)

cards_silver = cards_silver.withColumn(
    "credit_limit", col("credit_limit").cast("double")
)

In [ ]:
# Transformar transactions

transactions_silver = transactions_valid

for column in [
    "use_chip",
    "merchant_city",
    "merchant_state",
    "movement_type",
    "category",
    "direction",
    "income_source",
    "amount_convention",
]:
    transactions_silver = transactions_silver.withColumn(
        column, F.trim(F.lower(F.col(column)))
    )

for column in ["amount", "balance_before", "balance_after"]:
    transactions_silver = transactions_silver.withColumn(
        column, F.col(column).cast("decimal(18, 2)")
    )

## Convertir fechas

In [ ]:
# Convertir fecha de transactions

transactions_silver = transactions_silver.withColumn(
    "date",
    F.coalesce(
        F.try_to_timestamp(F.col("date").cast("string"), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col("date").cast("string"), F.lit("dd/MM/yyyy HH:mm")),
        F.try_to_timestamp(F.col("date").cast("string"), F.lit("MM/dd/yyyy HH:mm")),
    ),
)

## Eliminar registros duplicados

In [ ]:
# Eliminar duplicados

cards_silver = conservar_completo(cards_silver, "id")

transactions_silver = transactions_silver.dropDuplicates()

movimientos = transactions_silver.filter(
    F.col("amount_convention") == "signed_cashflow"
)


# No elegir arbitrariamente entre movimientos distintos con el mismo ID
if movimientos.groupBy("id").count().filter("count > 1").limit(1).count():
    raise ValueError("Hay movimientos diferentes con el mismo identificador.")

anteriores = transactions_silver.filter(
    F.col("amount_convention") == "legacy_unchanged"
)

if anteriores.join(movimientos.select("id"), "id", "inner").limit(1).count():
    raise ValueError("Un ID aparece en las dos convenciones de transacciones.")

otras = transactions_silver.filter(
    F.col("amount_convention").isNull()
    | ~F.col("amount_convention").isin("signed_cashflow", "legacy_unchanged")
)

anteriores = conservar_completo(anteriores, "id", ["date", "amount", "client_id"])

transactions_silver = movimientos.unionByName(anteriores).unionByName(otras)

## Validar cuentas y movimientos

Se revisan los identificadores, las referencias a clientes,
las convenciones de importes y la continuidad de los saldos.

Las transacciones anteriores con información esencial faltante
se separan como rechazadas.

Si un movimiento de cuenta presenta inconsistencias, se detiene
el procesamiento para evitar publicar un historial incompleto.

In [ ]:
# Definir validaciones


def exigir(nombre, df, condicion):
    errores = df.filter(~F.coalesce(condicion, F.lit(False)))

    if errores.limit(1).count():
        errores.show(5, truncate=False)

        raise ValueError(f"{nombre}: existen registros inconsistentes.")


def exigir_clave(nombre, df, clave):
    exigir(nombre, df, F.col(clave).isNotNull())

    if df.groupBy(clave).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{nombre}: existen claves duplicadas.")


def exigir_referencia(nombre, df, referencia, columnas):
    if (
        df.join(referencia.select(*columnas).distinct(), columnas, "left_anti")
        .limit(1)
        .count()
    ):
        raise ValueError(f"{nombre}: existen referencias inválidas.")


# Preparar cuentas

accounts_silver = accounts_spark.dropDuplicates()

for columna in ["opened_at", "balance_as_of"]:
    accounts_silver = accounts_silver.withColumn(
        columna, F.try_to_timestamp(F.col(columna), F.lit("yyyy-MM-dd HH:mm:ss"))
    )

exigir_clave("Accounts", accounts_silver, "account_id")

exigir_referencia("Accounts", accounts_silver, clientes, ["client_id"])

exigir(
    "Accounts",
    accounts_silver,
    F.col("opening_balance").isNotNull()
    & F.col("closing_balance").isNotNull()
    & F.col("currency").isNotNull()
    & (F.col("opened_at") <= F.col("balance_as_of")),
)


# Validar convenciones

exigir(
    "Transactions",
    transactions_silver,
    F.col("amount_convention").isin("legacy_unchanged", "signed_cashflow"),
)


# Separar registros anteriores incompletos

es_anterior = F.col("amount_convention") == "legacy_unchanged"

datos_completos = (
    F.col("id").isNotNull()
    & (F.col("id") >= 0)
    & F.col("date").isNotNull()
    & F.col("amount").isNotNull()
)

incompletos = transactions_silver.filter(
    es_anterior & ~F.coalesce(datos_completos, F.lit(False))
).withColumn("rejection_reason", F.lit("ID, fecha o importe inválido"))

# Homologar date para unirlo con los rechazados anteriores

transactions_rejected = transactions_rejected.withColumn(
    "date", F.col("date").cast("string")
).unionByName(
    incompletos.withColumn("date", F.col("date").cast("string")),
    allowMissingColumns=True,
)

transactions_silver = transactions_silver.filter(
    ~es_anterior | F.coalesce(datos_completos, F.lit(False))
)

transactions_rejected.write.mode("overwrite").parquet(
    f"{OUTPUT_PATH}/rejected/transactions.parquet"
)


# Seleccionar movimientos de cuenta

movements_silver = transactions_silver.filter(
    F.col("amount_convention") == "signed_cashflow"
)

exigir_clave("Movements", movements_silver, "id")

exigir_referencia(
    "Movements",
    movements_silver,
    accounts_silver,
    ["account_id", "client_id", "currency"],
)

exigir(
    "Movements",
    movements_silver,
    (F.col("id") >= 0)
    & F.col("date").isNotNull()
    & F.col("amount").isNotNull()
    & F.col("balance_before").isNotNull()
    & F.col("balance_after").isNotNull()
    & F.col("movement_type").isin(
        "income", "expense", "debt_payment", "transfer_in", "transfer_out"
    )
    & (F.col("balance_after") == F.col("balance_before") + F.col("amount")),
)

es_entrada = F.col("movement_type").isin("income", "transfer_in")

exigir(
    "Dirección de movimientos",
    movements_silver,
    F.when(
        es_entrada, (F.col("amount") > 0) & (F.col("direction") == "inflow")
    ).otherwise((F.col("amount") < 0) & (F.col("direction") == "outflow")),
)


# Validar continuidad de saldos

orden = Window.partitionBy("account_id").orderBy("date", "id")

orden_inverso = Window.partitionBy("account_id").orderBy(
    F.col("date").desc(), F.col("id").desc()
)

revision_saldos = (
    movements_silver.withColumn("previous_balance", F.lag("balance_after").over(orden))
    .withColumn("first_position", F.row_number().over(orden))
    .withColumn("last_position", F.row_number().over(orden_inverso))
    .join(
        accounts_silver.select(
            "account_id",
            "opening_balance",
            "closing_balance",
            "opened_at",
            "balance_as_of",
        ),
        "account_id",
    )
)

exigir(
    "Continuidad de saldos",
    revision_saldos,
    (
        F.col("balance_before")
        == F.when(F.col("first_position") == 1, F.col("opening_balance")).otherwise(
            F.col("previous_balance")
        )
    )
    & (
        (F.col("last_position") != 1)
        | (F.col("balance_after") == F.col("closing_balance"))
    )
    & (F.col("date") >= F.col("opened_at"))
    & (F.col("date") <= F.col("balance_as_of")),
)


# Validar cuentas sin movimientos

sin_movimientos = accounts_silver.join(
    movements_silver.select("account_id").distinct(), "account_id", "left_anti"
)

exigir(
    "Cuentas sin movimientos",
    sin_movimientos,
    F.col("opening_balance") == F.col("closing_balance"),
)


# Validar transferencias

es_transferencia = F.col("movement_type").isin("transfer_in", "transfer_out")

exigir(
    "Identificadores de transferencias",
    movements_silver,
    F.when(
        es_transferencia,
        F.col("transfer_id").isNotNull() & F.col("counterparty_account_id").isNotNull(),
    ).otherwise(
        F.col("transfer_id").isNull() & F.col("counterparty_account_id").isNull()
    ),
)

transferencias = movements_silver.filter(es_transferencia)

revision_transferencias = transferencias.groupBy("transfer_id").agg(
    F.count("*").alias("legs"),
    F.sum("amount").alias("net_amount"),
    F.countDistinct("account_id").alias("accounts"),
    F.countDistinct("date").alias("dates"),
    F.countDistinct("currency").alias("currencies"),
    F.countDistinct("movement_type").alias("types"),
    F.sort_array(F.collect_set("account_id")).alias("owners"),
    F.sort_array(F.collect_set("counterparty_account_id")).alias("counterparties"),
)

exigir(
    "Transferencias",
    revision_transferencias,
    (F.col("legs") == 2)
    & (F.col("net_amount") == 0)
    & (F.col("accounts") == 2)
    & (F.col("dates") == 1)
    & (F.col("currencies") == 1)
    & (F.col("types") == 2)
    & (F.col("owners") == F.col("counterparties")),
)

print("Cuentas y movimientos aprobados")

## Validar obligaciones y pagos

Se convierten las fechas y se revisa la relación entre cuotas,
cuentas y movimientos de pago.

Los pagos registrados deben coincidir con el importe, cliente,
cuenta y fecha del movimiento correspondiente.

Los días de atraso se calculan respecto a la fecha de corte.

In [ ]:
# Preparar obligaciones

obligations_silver = obligations_spark.dropDuplicates()

for columna in ["contract_start_date", "due_date", "payment_date", "as_of_date"]:
    obligations_silver = obligations_silver.withColumn(
        columna, F.to_date(F.try_to_timestamp(F.col(columna), F.lit("yyyy-MM-dd")))
    )

exigir_clave("Obligations", obligations_silver, "obligation_id")

exigir_referencia(
    "Obligations",
    obligations_silver,
    accounts_silver,
    ["account_id", "client_id", "currency"],
)

exigir(
    "Importes de obligaciones",
    obligations_silver,
    F.col("contract_id").isNotNull()
    & (F.col("installment_number") > 0)
    & (F.col("contract_principal") > 0)
    & (F.col("annual_interest_rate") >= 0)
    & (F.col("amount_due") > 0)
    & (F.col("amount_paid") >= 0)
    & (F.col("amount_paid") <= F.col("amount_due"))
    & (F.col("outstanding_amount") == F.col("amount_due") - F.col("amount_paid"))
    & (F.col("contract_start_date") <= F.col("due_date"))
    & (F.col("due_date") <= F.col("as_of_date")),
)

tiene_pago = F.col("amount_paid") > 0

exigir(
    "Fechas de pago",
    obligations_silver,
    F.when(
        tiene_pago,
        F.col("payment_transaction_id").isNotNull()
        & (F.col("payment_date") >= F.col("contract_start_date"))
        & (F.col("payment_date") <= F.col("as_of_date")),
    ).otherwise(
        F.col("payment_transaction_id").isNull() & F.col("payment_date").isNull()
    ),
)


# Comprobar movimientos de pago

pagos = movements_silver.filter(F.col("movement_type") == "debt_payment").select(
    F.col("id").alias("payment_transaction_id"),
    "obligation_id",
    "account_id",
    "client_id",
    "currency",
    (-F.col("amount")).alias("amount_paid"),
    F.to_date("date").alias("payment_date"),
)

claves_pago = [
    "payment_transaction_id",
    "obligation_id",
    "account_id",
    "client_id",
    "currency",
    "amount_paid",
    "payment_date",
]

obligaciones_pagadas = obligations_silver.filter(tiene_pago)

exigir_referencia("Pagos de obligaciones", obligaciones_pagadas, pagos, claves_pago)

exigir_referencia("Movimientos de pago", pagos, obligaciones_pagadas, claves_pago)

exigir_clave("Pago por obligación", pagos, "obligation_id")


# Calcular atraso a la fecha de corte

obligations_silver = obligations_silver.withColumn(
    "days_past_due_as_of",
    F.when(
        F.col("outstanding_amount") > 0,
        F.greatest(F.datediff("as_of_date", "due_date"), F.lit(0)),
    ).otherwise(0),
).withColumn(
    "days_to_full_payment",
    F.when(
        F.col("amount_paid") == F.col("amount_due"),
        F.greatest(F.datediff("payment_date", "due_date"), F.lit(0)),
    ),
)

print("Obligaciones y pagos aprobados")

## Preparar moneda y categorías

Se comprueba que los cinco datasets utilicen pesos mexicanos.

Los gastos se organizan en categorías generales.
Las categorías de gasto no reconocidas se agrupan en otros.

Los ingresos, transferencias y pagos de deuda conservan
su clasificación.

In [ ]:
# Normalizar categorías de gastos

categorias_gasto = [
    "alimentación",
    "transporte",
    "vivienda",
    "servicios",
    "salud",
    "entretenimiento",
    "compras",
    "viajes",
    "otros",
]

transactions_silver = transactions_silver.withColumn(
    "category",
    F.when(
        F.col("movement_type") == "expense",
        F.when(F.col("category").isin(*categorias_gasto), F.col("category")).otherwise(
            "otros"
        ),
    ).otherwise(F.col("category")),
)


# Validar la moneda de los cinco datasets
for nombre, datos in [
    ("users", users_silver),
    ("cards", cards_silver),
    ("transactions", transactions_silver),
    ("accounts", accounts_silver),
    ("obligations_payments", obligations_silver),
]:
    if (
        datos.filter(F.col("currency").isNull() | (F.col("currency") != "MXN"))
        .limit(1)
        .count()
    ):
        raise ValueError(f"{nombre}: se esperaba moneda MXN.")

print("Moneda y categorías preparadas")

## Imputar valores faltantes con Machine Learning

Se utiliza un modelo de vecinos cercanos para estimar los límites de crédito faltantes.

El modelo utiliza características de las tarjetas y la información financiera de los clientes.

Se compara el resultado con una imputación por mediana para seleccionar el método con menor error.

In [ ]:
# Importar librerías

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pyspark.sql.types import StructType, StructField, LongType, DoubleType

## Identificar la procedencia de los límites de crédito

Se inicializan las columnas que indican si un límite fue estimado
y qué método se utilizó.

Estas columnas se conservan incluso cuando no hay límites para imputar
o no existen valores conocidos suficientes para realizar la imputación.

In [ ]:
# Identificar límites observados y estimados

cards_silver = cards_silver.withColumn("credit_limit_imputed", lit(False)).withColumn(
    "credit_limit_imputation_method", lit(None).cast("string")
)

## Preparar los datos de entrenamiento

Se unen las características de las tarjetas con la información de los usuarios.

Los identificadores se utilizan para relacionar los registros, pero no forman parte de las variables predictoras.

In [ ]:
# Preparar información de usuarios

users_features = users_silver.select(
    col("id").alias("client_id"),
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score",
)


# Unir información

cards_features = cards_silver.select(
    "id",
    "client_id",
    "card_brand",
    "card_type",
    "has_chip",
    "num_cards_issued",
    "credit_limit",
).join(users_features, "client_id", "left")


# Convertir datos a pandas

cards_pd = cards_features.toPandas()


# Definir variables

numeric_columns = [
    "num_cards_issued",
    "current_age",
    "yearly_income",
    "total_debt",
    "credit_score",
]

categorical_columns = ["card_brand", "card_type", "has_chip"]

feature_columns = numeric_columns + categorical_columns


# Preparar valores faltantes

cards_pd[numeric_columns] = (
    cards_pd[numeric_columns].astype("float64").replace([np.inf, -np.inf], np.nan)
)

for column in categorical_columns:
    cards_pd[column] = cards_pd[column].map(
        lambda value: (
            np.nan
            if pd.isna(value) or str(value).strip() == ""
            else str(value).strip().lower()
        )
    )


# Separar registros

known_cards = cards_pd.loc[
    cards_pd["credit_limit"].notna() & cards_pd["client_id"].notna()
].copy()

missing_cards = cards_pd.loc[
    cards_pd["credit_limit"].isna()
    & cards_pd["id"].notna()
    & cards_pd["client_id"].notna()
].copy()

print("Tarjetas con límite conocido:", len(known_cards))
print("Tarjetas con límite faltante:", len(missing_cards))

## Crear el modelo de vecinos cercanos

Se normalizan las variables numéricas y se codifican las variables
categóricas.

Los clientes se separan entre entrenamiento y validación para evitar
que las tarjetas de un mismo cliente aparezcan en ambos conjuntos.

El modelo se entrena cuando hay límites para imputar, al menos dos
clientes con límites conocidos y cinco registros de entrenamiento.
Si no se cumplen las condiciones para entrenar KNN, se conserva
la mediana como método de imputación.

In [ ]:
# Separar variables

X = known_cards[feature_columns]
y = known_cards["credit_limit"]

groups = known_cards["client_id"]

selected_method = "mediana"
model = None


# Entrenar solo cuando existen límites para imputar y datos suficientes

if not missing_cards.empty and not known_cards.empty and groups.nunique() >= 2:
    # Dividir datos

    split = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)

    train_index, test_index = next(split.split(X, y, groups=groups))

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    if len(X_train) >= 5:
        # Preparar variables numéricas

        numeric_pipeline = Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
                ("scaler", StandardScaler()),
            ]
        )

        # Preparar variables categóricas

        categorical_pipeline = Pipeline(
            [
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent", keep_empty_features=True),
                ),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]
        )

        # Crear transformaciones

        preprocessor = ColumnTransformer(
            [
                ("numeric", numeric_pipeline, numeric_columns),
                ("categorical", categorical_pipeline, categorical_columns),
            ]
        )

        # Crear modelo

        model = Pipeline(
            [
                ("preprocessor", preprocessor),
                ("knn", KNeighborsRegressor(n_neighbors=5, weights="distance")),
            ]
        )

        # Entrenar modelo

        model.fit(X_train, y_train)

        print("Modelo entrenado")

## Evaluar el modelo

Cuando se ha entrenado KNN, se calcula el error absoluto medio
y se compara con la mediana de los datos de entrenamiento.

Se selecciona KNN únicamente si obtiene un error menor.
En caso contrario, se utiliza la mediana.

In [ ]:
if model is not None:
    # Generar predicciones

    knn_predictions = model.predict(X_test)

    median_predictions = np.full(len(y_test), y_train.median())

    # Calcular errores

    knn_mae = mean_absolute_error(y_test, knn_predictions)

    median_mae = mean_absolute_error(y_test, median_predictions)

    # Comparar resultados

    results = pd.DataFrame({"modelo": ["KNN", "Mediana"], "MAE": [knn_mae, median_mae]})

    display(results)

    # Seleccionar método

    selected_method = "knn" if knn_mae < median_mae else "mediana"


if not missing_cards.empty and not known_cards.empty:
    print("Método seleccionado:", selected_method)

## Completar los límites de crédito

Se aplica el método seleccionado a las tarjetas con límite faltante.

Si se seleccionó KNN, se vuelve a entrenar con todos los registros
conocidos antes de generar las estimaciones.

Se conservan las columnas que identifican los límites estimados
y el método utilizado.

Si no existen límites conocidos, los valores faltantes permanecen
sin imputar.

In [ ]:
# Estimar valores faltantes

if missing_cards.empty:
    print("No hay límites de crédito para imputar")

elif known_cards.empty:
    print("No hay límites de crédito conocidos para imputar")

else:
    if selected_method == "knn":
        model.fit(X, y)

        predictions = model.predict(missing_cards[feature_columns])

    else:
        predictions = np.full(len(missing_cards), y.median())

    # Preparar estimaciones

    estimated_cards = missing_cards[["id", "client_id"]].copy()

    estimated_cards["credit_limit_estimated"] = np.round(predictions, 2)

    display(estimated_cards)

    # Convertir estimaciones a Spark

    estimated_schema = StructType(
        [
            StructField("id", LongType(), False),
            StructField("credit_limit_estimated", DoubleType(), False),
        ]
    )

    estimated_rows = [
        (int(row.id), float(row.credit_limit_estimated))
        for row in estimated_cards.itertuples(index=False)
    ]

    estimated_spark = spark.createDataFrame(estimated_rows, schema=estimated_schema)

    # Actualizar cards

    cards_silver = (
        cards_silver.join(estimated_spark, "id", "left")
        .withColumn(
            "credit_limit_imputed",
            col("credit_limit").isNull() & col("credit_limit_estimated").isNotNull(),
        )
        .withColumn(
            "credit_limit_imputation_method",
            when(col("credit_limit_imputed"), lit(selected_method)).otherwise(
                lit(None).cast("string")
            ),
        )
        .withColumn(
            "credit_limit", coalesce(col("credit_limit"), col("credit_limit_estimated"))
        )
        .drop("credit_limit_estimated")
    )

    print("Límites de crédito completados")

## Verificar valores faltantes

In [ ]:
# Verificar valores faltantes

users_nulls = users_silver.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in users_silver.columns]
)

cards_nulls = cards_silver.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in cards_silver.columns]
)

transactions_nulls = transactions_silver.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in transactions_silver.columns]
)

print("Valores faltantes en users:")
users_nulls.show()

print("Valores faltantes en cards:")
cards_nulls.show()

print("Valores faltantes en transactions:")
transactions_nulls.show()

## Guardar SILVER

In [ ]:
# Guardar SILVER

datasets = {
    "users": users_silver,
    "cards": cards_silver,
    "transactions": transactions_silver,
    "accounts": accounts_silver,
    "obligations_payments": obligations_silver,
}

for nombre, datos in datasets.items():
    datos.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/{nombre}.parquet")

print("Pipeline SILVER completado")

In [ ]:
# Finalizar Spark

spark.stop()